# Step 5: Recover forenames from the Enhanced Author Names data

The PKG assigns gender only to authors with a full forename. For author rows coded `INI`/`UNK`, the forename is recovered
from the Enhanced Author Names (MEDLINE; Zhang et al. 2024): each PKG author row is matched to the MEDLINE author of the
same paper (`PMID`) with the same normalised surname. If the surname occurs more than once on the paper, the candidate with
the same **first initial** (PKG `ForeName` vs. MEDLINE initials) is preferred, then the one closest to the PKG `AuthorOrder`.
Position is only the last tie-break because PKG and MEDLINE author positions agree for only ~40 % of unique matches.
`AuthorNum` is not used for matching: it is not the position on the paper (in about a third of the rows it equals the
number of authors), and a merge on `PMID` + `AuthorNum` attaches the forename of a different co-author in about three
quarters of the cases (section 4).

**Input:** `data/derived/authors_gender_nan_ini.tsv` (PKG author rows without F/M; step 4), `data/raw/MEDLINE-enhanced-author-names-v2.tsv` (Zhang et al. 2024)
**Output:** `data/derived/nan_ini_medline_names.csv` – columns `AID, LastName, ForeName_pkg, gender, AuthorNum, AuthorOrder,
PMID, medline_name, forename` (the PKG forename column is renamed because DuckDB column names are case-insensitive) plus
match information: `match_n_candidates` (MEDLINE authors with the same surname on the paper), `match_n_initial` (of these,
with the same first initial; > 1 = still ambiguous), `match_pos_diff`, `match_initial_ok`.

In [ ]:
import time
import duckdb

import os

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RAW_DIR = f'{BASE_DIR}/data/raw'
DERIVED_DIR = f'{BASE_DIR}/data/derived'
PKG_TSV = f'{DERIVED_DIR}/authors_gender_nan_ini.tsv'                  # output of step 4
MEDLINE_TSV = f'{RAW_DIR}/MEDLINE-enhanced-author-names-v2.tsv'       # Enhanced Author Names (Zhang et al. 2024)
OUT_CSV = f'{DERIVED_DIR}/nan_ini_medline_names.csv'
DB_PATH = f'{DERIVED_DIR}/recover_forenames.duckdb'

con = duckdb.connect(DB_PATH)
con.execute("SET preserve_insertion_order = false")
# Memory: DuckDB's default limit (80 % of RAM) can leave too little headroom on machines with ~16 GB RAM; with a lower limit and
# fewer threads it writes intermediate results to disk (`<db>.tmp`) instead of failing. Raise both on a larger machine.
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")
t0 = time.time()
def log(msg):
    print(f'[{time.time() - t0:6.0f}s] {msg}')

## 1. Surname normalisation

PKG surnames contain HTML entities (`&Zbreve;itnik`) and leading blanks; MEDLINE surnames are lower case ASCII.
Both are normalised to lower-case ASCII letters only (entities reduced to their base letter, accents stripped).

In [ ]:
con.execute(r"""
CREATE OR REPLACE MACRO norm_name(x) AS
    regexp_replace(lower(strip_accents(regexp_replace(trim(x), '&([A-Za-z])[A-Za-z]*;', '\1', 'g'))), '[^a-z]', '', 'g')
""")
print(con.execute("SELECT norm_name(' &Zbreve;itnik') AS a, norm_name('Santana Armas') AS b, norm_name('Müller-Lüdenscheidt') AS c").df())

## 2. Load both files

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE pkg AS
SELECT
    row_number() OVER ()                                  AS rid,
    CAST(AID AS BIGINT)                                   AS AID,
    LastName, ForeName, gender,
    CAST(AuthorNum AS INTEGER)                            AS AuthorNum,
    CAST(AuthorOrder AS INTEGER)                          AS AuthorOrder,
    CAST(TRY_CAST(PMID AS DOUBLE) AS BIGINT)              AS PMID,
    norm_name(LastName)                                   AS ln,
    nullif(left(norm_name(ForeName), 1), '')              AS ini1   -- first initial (NULL if no forename)
FROM read_csv('{PKG_TSV}', delim = '\t', header = true, all_varchar = true)
""")
con.execute(f"""
CREATE OR REPLACE TABLE med AS
SELECT
    CAST(pm_id AS BIGINT)                                      AS pmid,
    CAST(author_position AS INTEGER)                           AS pos,
    enhanced_lastname_firstname_initials                       AS medline_name,
    norm_name(split_part(enhanced_lastname_firstname_initials, '|', 1)) AS ln,
    split_part(enhanced_lastname_firstname_initials, '|', 2)   AS forename,
    nullif(left(norm_name(coalesce(nullif(split_part(enhanced_lastname_firstname_initials, '|', 3), ''),
                                   split_part(enhanced_lastname_firstname_initials, '|', 2))), 1), '') AS ini1
FROM read_csv('{MEDLINE_TSV}', delim = '\t', header = true, all_varchar = true)
""")
log(con.execute("SELECT (SELECT count(*) FROM pkg) AS pkg_rows, (SELECT count(*) FROM med) AS medline_rows").df().to_string(index=False))

## 3. Match on PMID + surname (if the surname occurs several times on the paper: same first initial first, then closest position)

In [ ]:
# Memory: the join PKG x MEDLINE (~36M matched rows plus all candidates) does not fit into
# 16 GB of memory in one go. It is therefore run in N_BATCHES slices of PMIDs (PMID % N_BATCHES = k); every author row
# and all its candidates lie on the same PMID, so each slice is complete and the result is identical to one big query.
# Tie-break within a paper: 1) same first initial, 2) initial unknown, 3) different initial; then closest position.
N_BATCHES = 20          # increase (e.g. 40) if a slice still runs out of memory

con.execute("""
CREATE OR REPLACE TABLE matched (
    rid BIGINT, medline_name VARCHAR, forename VARCHAR,
    match_n_candidates BIGINT, match_n_initial BIGINT, match_pos_diff INTEGER, match_initial_ok BOOLEAN)
""")
for k in range(N_BATCHES):
    con.execute(f"""
    INSERT INTO matched
    SELECT
        p.rid, m.medline_name, m.forename,
        count(*) OVER (PARTITION BY p.rid)                                    AS match_n_candidates,
        count(*) FILTER (WHERE m.ini1 = p.ini1) OVER (PARTITION BY p.rid)    AS match_n_initial,
        abs(m.pos - p.AuthorOrder)                                            AS match_pos_diff,
        (m.ini1 = p.ini1)                                                     AS match_initial_ok   -- NULL if an initial is missing
    FROM (SELECT * FROM pkg WHERE PMID % {N_BATCHES} = {k} AND ln <> '') p
    JOIN (SELECT * FROM med WHERE pmid % {N_BATCHES} = {k}) m
      ON m.pmid = p.PMID AND m.ln = p.ln
    QUALIFY row_number() OVER (
        PARTITION BY p.rid
        ORDER BY CASE WHEN m.ini1 = p.ini1 THEN 0 WHEN m.ini1 IS NULL OR p.ini1 IS NULL THEN 1 ELSE 2 END,
                 abs(m.pos - p.AuthorOrder), m.pos) = 1
    """)
    log(f'batch {k + 1}/{N_BATCHES} done')
log(con.execute("SELECT count(*) AS rows_matched FROM matched").df().to_string(index=False))

## 4. Diagnostics: match quality, and comparison with a merge on `AuthorNum`

In [ ]:
# comparison: merge on PMID + AuthorNum – also in PMID slices to stay within memory
authornum_rows, authornum_same = 0, 0
for k in range(N_BATCHES):
    n, same = con.execute(f"""
    SELECT count(*), coalesce(sum((m.ln = p.ln)::INT), 0)
    FROM (SELECT * FROM pkg WHERE PMID % {N_BATCHES} = {k}) p
    JOIN (SELECT * FROM med WHERE pmid % {N_BATCHES} = {k}) m ON m.pmid = p.PMID AND m.pos = p.AuthorNum
    """).fetchone()
    authornum_rows += n; authornum_same += same

diag = con.execute("""
SELECT
    (SELECT count(*) FROM pkg)                                                     AS pkg_rows,
    (SELECT count(*) FROM pkg WHERE PMID IN (SELECT DISTINCT pmid FROM med))       AS rows_paper_in_medline,
    (SELECT count(*) FROM matched)                                                 AS rows_matched,
    (SELECT avg((match_n_candidates > 1)::INT) FROM matched)                       AS share_duplicate_surname,
    (SELECT avg((match_n_initial = 1)::INT) FROM matched WHERE match_n_candidates > 1) AS dup_resolved_by_initial,
    (SELECT avg(match_initial_ok::INT) FROM matched WHERE match_initial_ok IS NOT NULL) AS share_initial_agrees,
    (SELECT avg((match_pos_diff = 0)::INT) FROM matched WHERE match_n_candidates = 1)  AS share_same_position_unique
""").df()
diag['authornum_rows_with_name'] = authornum_rows
diag['authornum_share_same_surname'] = authornum_same / authornum_rows if authornum_rows else float('nan')
print(diag.T)

**Diagnostics (full run):**

| quantity | value | meaning |
|---|---|---|
| `pkg_rows` | 38.8 M | PKG author × paper rows without F/M (all candidates for recovery) |
| `rows_paper_in_medline` | 36.1 M (93 %) | paper is in the MEDLINE name file – upper bound for recovery |
| `rows_matched` | 35.8 M | 99.2 % of these find an author with the same surname on the same paper (92 % of all rows) |
| `share_duplicate_surname` | 6.3 % | matches where the surname occurs more than once on the paper |
| `dup_resolved_by_initial` | 86.4 % | of those, made unique by the first initial |
| `share_initial_agrees` | 99.8 % | first initial of PKG and matched MEDLINE name agree (where both are known) |
| `share_same_position_unique` | 40.3 % | unique matches whose MEDLINE position equals PKG `AuthorOrder` – why position is only the last tie-break |
| `authornum_rows_with_name` | 23.8 M (61 %) | rows that get a name when merging on `PMID` + `AuthorNum` |
| `authornum_share_same_surname` | 24.3 % | share of these names whose surname matches – i.e. ~76 % would belong to a co-author |

## 5. Export

In [ ]:
con.execute(f"""
COPY (
    SELECT p.AID, p.LastName, p.ForeName AS ForeName_pkg, p.gender, p.AuthorNum, p.AuthorOrder, p.PMID,
           m.medline_name, m.forename, m.match_n_candidates, m.match_n_initial, m.match_pos_diff, m.match_initial_ok
    FROM pkg p
    LEFT JOIN matched m USING (rid)
    ORDER BY p.rid
) TO '{OUT_CSV}' (HEADER, DELIMITER ',')
""")
log('written ' + OUT_CSV)
con.close()